# T2 · The nonlinear pendulum

Run from top to bottom with NumPy, PyTorch, Matplotlib and a Jupyter kernel. The reference trajectory is available to the evaluator. Only the sampled context is an observation.

## 1. From forces to motion
A point mass on a rod is pulled back by the tangential component of gravity. With $g/L=1$, the angular acceleration is $-\sin\theta$: near equilibrium it is almost proportional to displacement, but near the inverted position the restoring torque becomes small. The angle is unwrapped; it is not reduced modulo $2\pi$.

## 2. What fixes one realization?
The state is angle $\theta$ and angular velocity $p$: $\dot\theta=p,\ \dot p=-\sin\theta$. A position alone cannot tell us whether the bob is arriving or leaving. One initial pair and the equation fix the entire trajectory. ID proposals use $\theta_0\in[0.1,2.8]$, OOD uses $[2.9,3.1]$, both with $p_0\in[-0.5,0.5]$. The energy criterion below applies to both.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'worldlab/datasets/pendulum.py').exists())
sys.path.insert(0, str(root))
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from worldlab.datasets.pendulum import PendulumDataset as Dataset, make_realization
from worldlab.datasets._dynamics_common import N_OBS, DT_REF
position, velocity = 'theta', 'p'
ds = Dataset(split='train')
r = make_realization(seed=ds.seeds[0])
repeat = make_realization(seed=r.seed)
assert r.params == repeat.params and all(getattr(r,k).tobytes() == getattr(repeat,k).tobytes() for k in ('t_ref','z','z_t'))
print('seed', r.seed, 'parameters', r.params, 'reference shape/dtype', r.z.shape, r.z.dtype)
print('views per realization:', N_OBS, 'reference spacing (s):', DT_REF)

## 3. Validate the simulator
Classical RK4 combines four evaluations of the nonlinear derivative per step. Energy conservation alone would not detect an incorrect period. We check the small-angle limit, the amplitude dependence of the period, and the reduction in error when the step is halved.

In [ ]:
from worldlab.datasets.pendulum import integrate, rejection_stats
def energy(x,p):
    return p**2/2 + (1-np.cos(x))
t,x,p = integrate(theta0=0.1,p0=0)
rmse = np.sqrt(np.mean((x-0.1*np.cos(t))**2))
assert rmse < 1e-3
print('Small-angle RMSE (physical linearization error):', rmse)
fig, (ax, de) = plt.subplots(1,2,figsize=(11,3.3))
periods = []
for x0 in (0.1,0.2,1.0,2.8):
    t,x,p = integrate(theta0=x0,p0=0)
    E = energy(x,p)
    drift = abs(E/E[0]-1)
    assert drift.max() < 1e-10
    de.plot(t,drift,label=f'{x0} rad')
    if x0 != 0.1:
        ix = np.flatnonzero((x[:-1]>0) & (x[1:]<=0))
        crossings = t[ix]-DT_REF*x[ix]/(x[ix+1]-x[ix])
        periods.append(np.diff(crossings).mean())
        ax.plot(t,x,label=f'{x0} rad: {periods[-1]:.3f} s')
    print(f'initial angle {x0}: maximum relative drift {drift.max():.2e}')
assert np.all(np.diff(periods)>0)
ax.set(xlabel='Time (s)',ylabel='Angle (rad)',title='Periods from descending zero crossings')
de.set(xlabel='Time (s)',ylabel='Absolute relative energy drift',title='Fixed step, no damping')
for panel in (ax,de): panel.legend()
fig.tight_layout(); plt.show()

At large amplitude the bob lingers near its turning point. To distinguish integration error from this physical change, compare **both** components against a finer nonlinear RK4 solution on shared grid points. Coarser diagnostic steps avoid roundoff saturation; production stays at 1 ms.

In [ ]:
h = np.array([0.04,0.02,0.01])
fine = 0.0025
_, xf, pf = integrate(theta0=1,p0=0.2,dt=fine)
errors = []
for step in h:
    _,x,p = integrate(theta0=1,p0=0.2,dt=step)
    stride = round(step/fine)
    errors.append(np.sqrt(np.mean((x-xf[::stride])**2+(p-pf[::stride])**2)))
slope = np.polyfit(np.log(h),np.log(errors),1)[0]
ratios = np.array(errors[:-1])/errors[1:]
assert np.all((14 < ratios) & (ratios < 18))
print('error ratios:', ratios, 'fitted order:', slope)
plt.loglog(h,errors,'o-',label='Measured joint RMS error')
plt.loglog(h,errors[0]*(h/h[0])**4,'--',label='Fourth-order reference')
plt.xlabel('Step (s)'); plt.ylabel('Error against fine solution'); plt.legend(); plt.show()

### Why reject an initial pair?
The top of the swing has potential energy 2. Below it the bob librates; above it the unwrapped angle can increase or decrease without bound. Equality is the separatrix and is excluded. For these angles the admissible interval is $|p_0|<2\cos(\theta_0/2)$. It shrinks near the top. Redrawing **both** coordinates preserves the joint proposal conditioned on libration, not independent uniform accepted marginals.

In [ ]:
for split in ('train','val','test','param_ood'):
    stats = rejection_stats(split)
    print(f"{split}: {stats['rejected']}/{stats['proposed']} proposals rejected ({stats['fraction']:.2%}); "
          f"{stats['changed_seeds']} seeds needed a redraw")
for seed in Dataset(split='param_ood').seeds:
    rng = np.random.default_rng(seed)
    pair = rng.uniform(2.9,3.1), rng.uniform(-0.5,0.5)
    if energy(*pair)>2: break
assert energy(*pair)>2
accepted = make_realization(seed=seed,param_ood=True)
assert energy(accepted.z[0],accepted.z_t[0]) < 2
for (x0,p0), label in [(pair,f'Rejected first proposal, seed {seed}'),((2.9,0),'Near-separatrix libration')]:
    t,x,p = integrate(theta0=x0,p0=p0)
    plt.plot(t,x,label=f'{label}: E={energy(x0,p0):.4f}')
plt.xlabel('Time (s)'); plt.ylabel('Unwrapped angle (rad)'); plt.legend(); plt.show()

## 4–5. What is observed, and what is queried?
Each realization has 32 independently seeded configurations per mode. A sensor records position, velocity, or both, without noise. The step is chosen from 0.05, 0.1, 0.2 and 0.5 seconds; the cutoff is uniform between 60% and 90% of the duration. The last sampled time (dashed line) is the hull boundary, usually **before** the cutoff (dotted line).

Each row uses its own real context, query and sensor channels; the first two share physical seed and configuration index. Mode enters the view seed, so changing mode can change the observations as well as the query. Parameter OOD changes the physical draw: here its query is inside its own hull.

In [ ]:
views = [ds[0], Dataset(split='train',mode='extrapolate')[0], Dataset(split='param_ood')[0]]
fig, axes = plt.subplots(3,2,figsize=(11,7),sharex=True)
for s, panels, color in zip(views,axes,['tab:green','tab:red','tab:purple']):
    sensor = s['sensor_type']
    channels = [sensor] if sensor in (position,velocity) else [position,velocity]
    assert s['t_max'] == s['t_i'][-1] < s['t_obs']
    assert s['d_q'] == max(0,s['t_q']-s['t_max'])
    if s['mode']=='extrapolate': assert s['t_q'] >= s['t_obs']
    else: assert 0 <= s['t_q'] <= s['t_max']
    for ax,ch,field in zip(panels,(position,velocity),('z','z_t')):
        ax.plot(s['t_ref'],s[field],color='0.7',label='Full truth')
        if ch in channels:
            j=channels.index(ch)
            ax.scatter(s['t_i'],s['values'][:,j],s=9,label='Observed')
            ax.scatter(s['t_q'],s['target'][j],marker='*',s=110,color=color,label='Query target')
        ax.axvline(s['t_max'],ls='--',color='k'); ax.axvline(s['t_obs'],ls=':',color='0.5')
        ax.set_ylabel(ch); ax.legend(fontsize=7)
    print(f"{s['mode']:12s} seed={s['seed']}  sensor={sensor}  t_max={s['t_max']:.3f}  "
          f"cutoff={s['t_obs']:.3f}  query={s['t_q']:.3f}  d_q={s['d_q']:.3f}")
assert views[0]['realization_id']==views[1]['realization_id']
for ax in axes[-1]: ax.set_xlabel('Time (s)')
fig.tight_layout(); plt.show()

T2 selects observation and query values by exact reference-grid indices. Interpolation here describes the **prediction task**, not a numerical interpolation of the target.

## 6. A real training batch
Inspect one item before batching. Pad time to the longest context and keep two named channels so that a velocity-only sensor is never mistaken for a position sensor. A Boolean mask distinguishes observed zeros from padding and missing channels. The reference trajectory is not an input to this batch.

In [ ]:
s=views[0]
print('Item keys:', ', '.join(s))
print('Arrays:', {k:(v.shape,str(v.dtype)) for k,v in s.items() if isinstance(v,np.ndarray)})
def collate(rows):
    lengths = torch.tensor([len(r['t_i']) for r in rows])
    B, L = len(rows), int(lengths.max())
    context = torch.zeros(B,L,2,dtype=torch.float64)
    times = torch.zeros_like(context[:,:,0])
    observed = torch.zeros_like(context,dtype=torch.bool)
    target = torch.zeros_like(context[:,0])
    target_mask = torch.zeros_like(observed[:,0])
    for i,r in enumerate(rows):
        n = len(r['t_i'])
        times[i,:n] = torch.from_numpy(r['t_i'])
        channels = {position:[0],velocity:[1]}.get(r['sensor_type'],[0,1])
        for j,c in enumerate(channels):
            context[i,:n,c] = torch.from_numpy(r['values'][:,j])
            observed[i,:n,c] = True
            target[i,c] = float(r['target'][j])
            target_mask[i,c] = True
    assert torch.equal(observed.any(dim=2).sum(dim=1), lengths)
    return dict(context=context,times=times,observed=observed,lengths=lengths,
                query=torch.tensor([r['t_q'] for r in rows],dtype=torch.float64),
                target=target,target_mask=target_mask)
batch = next(iter(DataLoader(ds,batch_size=4,shuffle=False,collate_fn=collate)))
print('Batch:', {k:tuple(v.shape) for k,v in batch.items()})
print('Unpadded lengths:', batch['lengths'].tolist())
assert batch['lengths'].unique().numel()>1

## What this testbed tests in NO-JEPA

T2 asks whether a context encodes nonlinear dynamics. A generic sinusoid may fit a short segment but miss the slow turn near the top; its timing error grows in the unobserved future. Multiple sensor views of the same trajectory let us ask whether the representation follows the physical state rather than the sampling schedule.

The OOD split tests initial angles outside training while keeping every trajectory oscillatory. Compare within-hull OOD predictions separately from long-horizon predictions. The acceptance condition also changes the distribution of accepted velocities: an OOD difference cannot be attributed to a uniform angle shift alone.